# Day 77: Image Classification with Transfer Learning
This notebook demonstrates a complete portfolio-ready image classification project using transfer learning (VGG16), data augmentation, and fine-tuning on the built-in `tf_flowers` dataset.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import VGG16
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import numpy as np


## Load TensorFlow Flowers Dataset
We will use the built-in `tf_flowers` dataset with 5 classes: Daisy, Dandelion, Roses, Sunflowers, Tulips.

In [ ]:
import tensorflow_datasets as tfds

# Load dataset
(dataset_train, dataset_val), info = tfds.load(
    'tf_flowers',
    split=['train[:80%]', 'train[80%:]'],
    with_info=True,
    as_supervised=True
)

NUM_CLASSES = info.features['label'].num_classes
IMG_SIZE = 224
BATCH_SIZE = 16

## Data Augmentation
We apply rotation, flipping, zoom, and color shift to improve generalization.

In [ ]:
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

# Apply preprocessing
dataset_train = dataset_train.map(preprocess).shuffle(1000).batch(BATCH_SIZE)
dataset_val = dataset_val.map(preprocess).batch(BATCH_SIZE)

# Data augmentation layers
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomContrast(0.1)
])

## Load VGG16 Pretrained Model
We remove the top layers and freeze the base for feature extraction.

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base_model.trainable = False  # Freeze base

# Build the model
model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')  # multi-class
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## Train classifier on top layers first (Feature Extraction)
We train only the new classifier layers initially.

In [ ]:
EPOCHS_FE = 5

history_fe = model.fit(
    dataset_train,
    validation_data=dataset_val,
    epochs=EPOCHS_FE
)

## Fine-Tuning Top Layers
Unfreeze some of the base model layers for better performance.

In [ ]:
# Unfreeze last few layers of VGG16
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

# Recompile with small learning rate
model.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

EPOCHS_FT = 5

history_ft = model.fit(
    dataset_train,
    validation_data=dataset_val,
    epochs=EPOCHS_FT
)

## Plot Training & Validation Accuracy

In [ ]:
acc = history_fe.history['accuracy'] + history_ft.history['accuracy']
val_acc = history_fe.history['val_accuracy'] + history_ft.history['val_accuracy']

plt.figure(figsize=(8,5))
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.title('Training & Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()
model.save("day77_flowers_classifier.keras")

## Make Predictions on Sample Images

In [ ]:
import matplotlib.pyplot as plt

for image_batch, label_batch in dataset_val.take(1):
    predictions = model.predict(image_batch)
    predicted_labels = np.argmax(predictions, axis=1)

    plt.figure(figsize=(12,8))
    for i in range(9):
        plt.subplot(3,3,i+1)
        plt.imshow(image_batch[i])
        plt.title(f"True: {label_batch[i].numpy()}, Pred: {predicted_labels[i]}")
        plt.axis('off')
    plt.show()